# Agent 

An agent is an AI system that can think, decide which tools to use, and take actions to solve a problem.

### What is an Agent?

    An Agent is a reasoning engine powered by an LLM that:

        Reads the user’s question,
        
        Decides which tools to use (if any),
        
        Calls them in sequence,
        
        Combines the results,
        
        Produces a final answer.
        

### Tool

A Tool is just a Python function that the LLM (agent) is allowed to call to perform an action.

    Think of a tool like a “superpower” you give to the model —

    it can use that function to get real data or perform calculations beyond its own text generation ability.

### Rules for Tool

The @tool decorator wraps it so that the LLM can see it as something it can use.

The docstring ("""Multiply two numbers.""") becomes the tool’s description — this tells the agent when to use it.



###  tools are the bridge between the LLM and the real world (Python code, APIs, databases, etc.).


| Role                       | Real-world example     | LangChain concept |
| -------------------------- | ---------------------- | ----------------- |
| You                        | A human problem-solver | **Agent**         |
| Calculator, phone, browser | External abilities     | **Tools**         |
| Your brain                 | Reasoning (LLM)        | **LLM**           |


### User → Agent (LLM thinks) → Uses Tool(s) → Gets Observation → Answers User


🧩 Agent = Brain

🧰 Tool = Hands

💬 User Query = Task


# First Agent - tool calling

### summary about agent


User → Agent (LLM thinks) → Uses Tool(s) → Gets Observation → Answers User



from langchain_core.tools import tool

The @tool decorator turns a normal Python function into a LangChain-compatible Tool object that agents can call.



### Without @tool



def multiply(a, b):
    return a * b

The LLM cannot see or call this function — it’s just a local Python function.

### With @tool


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

    The LLM sees it as a named tool: "multiply".
    
    It knows what inputs to provide (a, b).
    
    It reads your docstring ("""Multiply two numbers.""") to decide when to use it.
    

### First agent

In [8]:
!pip show langchain

Name: langchain
Version: 1.2.8
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\nvgra\anaconda3\annacond\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: embedchain


In [5]:
#pip show langchain_core

In [7]:
!pip show langgraph

Name: langgraph
Version: 1.0.7
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License: 
Location: C:\Users\nvgra\anaconda3\annacond\Lib\site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: langchain


In [9]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Tool
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# Local LLM
model = ChatOllama(model="llama3.2", temperature=0)

agent = create_react_agent(
    model,
    tools=[add_numbers]
)

# Run agent
response = agent.invoke(
    {"messages": [("user", "Add 12 and 30")]}
)

print(response["messages"][-1].content)


C:\Users\nvgra\AppData\Local\Temp\ipykernel_11820\102470142.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


The result of adding 12 and 30 is 42.


| Import               | Purpose                                                    |
| -------------------- | ---------------------------------------------------------- |
| `ChatOllama`         | Connects LangChain to your **local Ollama LLM** (llama3.2) |
| `tool`               | Converts a Python function into an **LLM-callable tool**   |
| `create_react_agent` | Creates a **ReAct-style agent** (Reason + Act + Observe)   |


1. @tool wraps the function

2.LangChain reads:

    Function name → add_numbers 
    
    Input schema → a: int, b: int
    
    Description → from docstring

3. This metadata is sent to the LLM so it knows:

    When to call the function
    
    What arguments it needs

Connects LangChain to Ollama running locally

Loads the model: llama3.2

temperature=0 → deterministic responses (best for tools)

Creating the ReAct Agent

agent = create_react_agent(
    model,
    tools=[add_numbers]
)

This is the MOST IMPORTANT STEP

Internally, LangGraph does:

    Registers the LLM
    
    Registers all tools
    
    Builds a ReAct loop:   Thought → Action → Observation → Thought → Final Answer

Why ReAct?

ReAct allows the LLM to:

    Reason about the question
    
    Decide to call a tool
    
    Use the tool’s output in the final answer

In [ ]:
print(response["messages"][-1].content)

   {
  "messages": [
    HumanMessage(...),
    AIMessage(thoughts...),
    ToolMessage(result=42),
    AIMessage(content="The sum of 12 and 30 is 42.")
  ]
}

Why [-1]?

The last message is always the final answer

Earlier messages contain reasoning + tool calls

# Second Agent - simple Greet

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# --- Local LLM ---mj
llm = ChatOllama(model="mistral", temperature=0)

# --- Tool ---
@tool
def greet(name: str) -> str:
    """Return a greeting message."""
    return f"Hello, {name}! Nice to meet you."

# --- Agent ---
agent = create_react_agent(llm, [greet])

# --- Stronger instruction ---
result = agent.invoke({
    "messages": [
        ("system", "You are a helpful assistant that must use tools whenever appropriate."),
        ("user", "Use the greet tool to greet my friend Dr Ganapathi Raju.")
    ]
})

print("\nAgent says:", result["messages"][-1].content)


C:\Users\nvgra\AppData\Local\Temp\ipykernel_11820\2504081747.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [greet])



Agent says:  It seems like the assistant has responded with a friendly greeting for your friend Dr. Ganapathi Raju.

[{"name": "greet", "arguments": {"name":"Dr Ganapathi Raju"}}]


In [11]:

print(greet.invoke({"name": "Dr Ganapathi Raju"}))

Hello, Dr Ganapathi Raju! Nice to meet you.


# Multi Tool Agent

In [12]:

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent  

# --- Create the local LLM (Ollama) ---
llm = ChatOllama(model="mistral", temperature=0)

# --- Define multiple tools ---
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# --- Register tools ---
tools = [add, subtract, multiply]

# --- Create the reasoning agent ---
agent = create_react_agent(llm, tools)

# --- Run the agent ---
result = agent.invoke({
    "messages": [("user", "Add 5 and 10, then multiply by 3.")]
})

# --- Print the agent’s final response ---
print("\nAgent says:", result["messages"][-1].content)


C:\Users\nvgra\AppData\Local\Temp\ipykernel_11820\2933352450.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)



Agent says:  It seems there was an issue with the previous response. The result of adding 5 and 10 is indeed 15, but when we multiply that by 3, the correct answer should be 45. I apologize for any confusion caused. Here's the corrected response:

[{"name": "add", "arguments": {"a":5,"b":10}}{"name": "multiply", "arguments": {"a":{"value":"result_of_previous_function"},"b":3}}]

[{"content": 45}]


### Tool + SQLite Database using LangGraph Agent

User: List students enrolled in AI
   ↓
LLM: Needs DB info
   ↓
LLM: Chooses find_students_by_course
   ↓
Tool executes SQL query
   ↓
Result returned
   ↓
LLM formats human-readable answer


In [ ]:
import sqlite3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    course TEXT
)
""")

cursor.executemany(
    "INSERT INTO students (name, age, course) VALUES (?, ?, ?)",
    [
        ("Anil", 21, "Data Science"),
        ("Meena", 22, "AI"),
        ("Ravi", 23, "ML"),
    ]
)

conn.commit()
conn.close()


In [ ]:
from langchain_core.tools import tool
import sqlite3

@tool
def get_all_students() -> str:
    """Fetch all students from the database."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name, age, course FROM students")
    rows = cursor.fetchall()
    conn.close()

    return "\n".join([f"{r[0]} ({r[1]}) - {r[2]}" for r in rows])


In [ ]:
@tool
def find_students_by_course(course: str) -> str:
    """Find students enrolled in a specific course."""
    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, age FROM students WHERE course = ?",
        (course,)
    )
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return "No students found."

    return "\n".join([f"{r[0]} ({r[1]})" for r in rows])


In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3.2",
    temperature=0
)


In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model,
    tools=[get_all_students, find_students_by_course]
)


In [ ]:

response = agent.invoke(
    {"messages": [("user", "Show all students")]}
)

print(response["messages"][-1].content)


In [ ]:
response = agent.invoke(
    {"messages": [("user", "List students enrolled in AI give correct info from db")]}
)

print(response["messages"][-1].content)


### chef agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# TOOL

@tool
def recipe_finder(ingredients: str) -> str:
    """
    Suggest recipes using given ingredients.
    """

    return f"""
Here are 3 recipes using: {ingredients}

1. Vegetable Omelette
   Beat eggs with chopped {ingredients}, cook in a pan.

2. Stir Fry
   Saute {ingredients} with oil, salt and spices.

3. Quick Soup
   Boil {ingredients} with garlic, pepper and water.
"""

# LOCAL MODEL

model = ChatOllama(
    model="llama3.2",
    temperature=0
)

# AGENT

agent = create_react_agent(
    model,
    tools=[recipe_finder]
)

# CHAT LOOP

while True:

    query = input("\nUser: ")

    if query.lower() == "exit":
        break

    response = agent.invoke(
        {"messages": [("user", query)]}
    )

    print("\nChef:", response["messages"][-1].content)

# Dynamic Multi-Tool Agent

In [ ]:

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent 

# --- Local LLM (Ollama) ---
llm = ChatOllama(model="mistral", temperature=0)

# --- Tools ---
@tool
def greet(name: str) -> str:
    """Greets a person by name."""
    return f"Hello, {name}! Nice to meet you."

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def weather(city: str) -> str:
    """Get the current temperature of a city (demo)."""
    
    return f"The weather in {city} is 29°C and sunny."

# --- Register tools ---
tools = [greet, add, weather]

# --- Create the ReAct Agent ---
agent = create_react_agent(llm, tools)

# --- Run queries ---
queries = [
    "Hi, greet my friend Mr Raju .",
    "Add 25 and 13.",
    "What's the weather in Hyderabad?",
]

for q in queries:
    result = agent.invoke({"messages": [("user", q)]})
    print(f"\n Query: {q}")
    print(" Agent:", result["messages"][-1].content)


# Agent with Memory

### ReAct Agent with Memory

Each query was independent.

The agent forgot everything after each question.


### Why Memory ?

### Without memory:

    Every time you ask a question, the agent starts from scratch.
    
    It forgets all previous context.

### With memory:
    
    The agent can recall what was said earlier.
    
    It can maintain conversations and chain reasoning across multiple turns.
    


### Where Memory Fits in the Agent Architecture

User → Agent → LLM → Tools
   ↕
 Memory (stores context)

The memory layer sits between the user and the LLM,

saving important context — like previous messages, results, or tool outputs.


###  Memory in LangGraph

LangGraph (newer framework built on top of LangChain) introduces a checkpoint system — instead of just storing text, it stores the entire agent state.

So in LangGraph, Memory = Checkpoint (saved conversation state).

### MemorySaver


from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

    Each chat thread is saved in a dictionary.
    
    You can have multiple sessions (thread_id).
    
    It auto-saves conversation history and intermediate steps.


# With out Memory

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent  

# --- Define a simple tool ---
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# --- Create local LLM (Ollama) ---
llm = ChatOllama(model="llama3.2", temperature=0)

# --- Register tools ---
tools = [add]

# --- Create a stateless ReAct agent (no memory) ---
agent = create_react_agent(llm, tools)


result1 = agent.invoke({"messages": [("user", "My name is Ganapathi.")]})
print("Response 1:", result1["messages"][-1].content)

result2 = agent.invoke({"messages": [("user", "Add 5 and 10.")]})
print("Response 2:", result2["messages"][-1].content)


In [ ]:

result2 = agent.invoke({"messages": [("user", "What is my name?")]})
print("Response 2:", result2["messages"][-1].content)


# With Memory

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.tools import tool

from langgraph.checkpoint.memory import MemorySaver

# --- Define Tool ---
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# --- Initialize Ollama LLM ---
# You can replace 'llama3' with 'mistral', 'gemma2', etc.
llm = ChatOllama(model="mistral", temperature=0)

# --- Initialize memory checkpointer ---
memory = MemorySaver()

# --- Create the Agent ---
agent = create_agent(llm, [add], checkpointer=memory)

# --- Persistent Session ID (for memory) ---
config = {"configurable": {"thread_id": "chat_1"}}

# --- Step 1: User introduces name ---
result1 = agent.invoke({"messages": [("user", "My name is Ganapathi.")]}, config=config)
print("Response 1:", result1["messages"][-1].content)

# --- Step 2: Ask again (agent remembers via MemorySaver) ---
result2 = agent.invoke({"messages": [("user", "What is my name?")]}, config=config)
print("Response 2:", result2["messages"][-1].content)


### Example with memory

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

# --- Define a Tool ---
@tool
def multiply(a: int, b: int) -> str:
    """Multiply two numbers."""
    return str(a * b)

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# --- Initialize Ollama LLM ---

llm = ChatOllama(model="llama3.2", temperature=0)

# --- Create memory checkpoint (for persistence) ---

memory = MemorySaver()

# --- Create the Agent ---

agent = create_agent(llm, [multiply,add], checkpointer=memory)

# --- Persistent Thread ID for memory ---

config = {"configurable": {"thread_id": "math_session_1"}}

# --- Step 1: First message ---

result = agent.invoke({"messages": [("user", "What is 5 * 6?")]}, config=config)

print("Response 1:", result["messages"][-1].content)

# --- Step 2: Ask follow-up (agent recalls previous answer) ---

result2 = agent.invoke({"messages": [("user", "Now add 10 to that result.")]}, config=config)

print("Response 2:", result2["messages"][-1].content)


# Custom Prompt


In normal LangChain agents (like ZERO_SHOT_REACT_DESCRIPTION), the reasoning style is predefined —

    it uses the ReAct framework ("Reason + Act + Observe + Final Answer").

But in a Custom Prompt Agent, you write your own instructions telling the LLM how to think.


# Custom Prompt Agent 

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


llm = ChatOllama(model="mistral", temperature=0)

# --- Create a structured reasoning prompt ---

prompt = ChatPromptTemplate.from_messages([
    ("system",
     
         "You are a calm, logical assistant. "
         "Follow these steps:\n"
         "1️ THINK carefully about the question.\n"
         "2️ PLAN your reasoning.\n"
         "3️ GIVE the final clear answer."),
    
    ("human", "{question}")
])

# --- Chain the prompt with the LLM ---

chain = prompt | llm

# --- Run the reasoning query ---

response = chain.invoke({"question": "Find the square of 12."})

print(" Agent says:", response.content)


# Custom Prompt + Tool

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# --- 1️. Define tools ---
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

tools = [add, multiply]

# --- 2️. Use local Ollama LLM ---

llm = ChatOllama(model="mistral", temperature=0)

# --- 3️. Custom reasoning prompt ---
prompt = ChatPromptTemplate.from_messages([
    ("system",
         "You are a calm, logical reasoning assistant.\n"
         "Follow these steps:\n"
         "1️ THINK about the question.\n"
         "2️ DECIDE if you need to use a tool.\n"
         "3️ CALL the correct tool if required.\n"
         "4️ Give the final answer clearly."),
    ("human", "{question}")
])

# --- 4️. Tool selection logic ---
def tool_router(question: str):
    
    q = question.lower()
    
    if "multiply" in q or "product" in q:
        
        return multiply.invoke({"a": 12, "b": 4})
        
    elif "add" in q or "sum" in q:
        
        return add.invoke({"a": 10, "b": 5})
        
    else:
        return "I can only add or multiply for now."

# --- 5️. Combine reasoning + tool logic ---

def reasoning_chain(inputs):
    
    question = inputs["question"]
    thought = llm.invoke(prompt.format(question=question))
    answer = tool_router(question)
    return f"{thought.content}\n\n Final Answer: {answer}"

# --- 6️. Run the reasoning agent ---

response = reasoning_chain({"question": "Multiply 12 and 4."})
print(" Agent says:\n", response)
